# EDA Recommender Multimodal

Exploratory analysis for multimodal recommender catalogs.

Steps:
- Inspect catalog files for size and schema.
- Summarize item counts and sample entries.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

rec_root = REPO_ROOT / 'data' / 'raw' / 'recommendation'
files = [
    'items.csv',
    'books.csv',
    'games.csv',
    'places.csv',
    'phones.csv',
    'laptops.csv',
    'headphones.csv',
    'cameras.csv',
]

summary = {'catalogs': {}}

print('Recommendation root:', rec_root)
for name in files:
    path = rec_root / name
    if not path.exists():
        print('Missing:', path)
        continue
    size_mb = round(path.stat().st_size / 1024**2, 2)
    print(f'{name}: {size_mb} MB')
    df = pd.read_csv(path)
    summary['catalogs'][name] = {
        'rows': int(df.shape[0]),
        'cols': int(df.shape[1]),
        'columns': list(df.columns),
    }
    print('  shape:', df.shape)
    print('  sample:', df.head(3).to_dict(orient='records'))


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_recommender_multimodal_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize recommender-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'recommender' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No recommender entries found in TRAINING_DATA.json')
    else:
        print('recommender datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
